# CUE 01: values, types, constraints

In CUE a type is a value with more possibilities and a concrete value is the most specific type. Unification (`&`) merges them; a conflict is an error, not an override. This lab lives in `/source/work/cue-lab`.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && rm -rf cue-lab && mkdir cue-lab && cd cue-lab && cue mod init example.com/lab && cat cue.mod/module.cue


In [ ]:
cd /source/work/cue-lab
cat > schema.cue <<'CUE'
package lab

#Context: {
    env: "dev" | "prod"
    web: {
        image:    *"traefik/whoami:v1.11.0" | (string & !~":latest$")
        replicas: int & >0 & <=20
        motd:     string | *"hello from cue"
    }
    host: "web.\(domain)"
    domain: string
}
CUE
cat > data.cue <<'CUE'
package lab

dev: #Context & {env: "dev", domain: "dev.example.com", web: replicas: 1}
prod: #Context & {env: "prod", domain: "example.com", web: {replicas: 3, motd: "hello from prod"}}
CUE
cue fmt ./... && cue vet ./... && echo "vet: ok" && cue eval . -e prod


Defaults (`*`) fill what a context leaves out, string interpolation derives `host` from `domain`, and the export is plain data.


In [ ]:
cd /source/work/cue-lab
cue export . -e dev --out yaml


In [ ]:
cd /source/work/cue-lab
cat > bad.cue <<'CUE'
package lab

prod: web: replicas: 30
CUE
(cue vet ./... 2>&1 | head -4) || true; rm bad.cue


The error names the constraint and both source positions. That is the feedback loop the talk keeps coming back to: for people and for agents.


In [ ]:
cd /source/work/cue-lab
cue def . -e '#Context'


Try it: add a constraint that rejects an image without a registry host, then vet the contexts.
